In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [89]:
from langchain_deepseek import ChatDeepSeek


llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.5,
    top_p=0.6,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

llm

ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x7c80b0703e30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7c80b08f85f0>, root_client=<openai.OpenAI object at 0x7c80b07acef0>, root_async_client=<openai.AsyncOpenAI object at 0x7c80b07ad970>, model_name='deepseek-chat', temperature=0.5, model_kwargs={}, max_retries=2, top_p=0.6, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1')

In [58]:
template = '''
You are a regular user of the Rutube video platform, where you watch movies, series, cartoons, shows, and live broadcasts.
You’ve just received the following informational text in Russian (about Rutube's features, content, updates, or services).

Think like an actual user reading this message: what would you want to know or clarify after reading it?

Come up with 5 natural and realistic questions in the first person that could be answered directly by the content of the text.
Avoid overly generic or technical questions—focus on what a real user would likely ask in this situation.

Output ONLY the questions in valid JSON format, following this structure:
{
  "questions": [
    "Первый вопрос",
    "Второй вопрос",
    "Третий вопрос",
    "Четвертый вопрос",
    "Пятый вопрос"
  ]
}
'''

In [49]:
text = "Скрипт ответа на предложения пользователя/вопрос о функционале, который отсутствует. Мы постоянно работаем над улучшением платформы и прислушиваемся к мнению пользователей. Ваше пожелание уже передали коллегам."

prompt = template + "\n" + text

llm.invoke(prompt).text()

'{\n  "questions": [\n    "Когда появится возможность скачивать видео для просмотра офлайн?",\n    "Можно ли создать отдельный профиль для ребенка с ограничением по возрасту?",\n    "Будут ли добавлять больше сериалов в оригинальной озвучке?",\n    "Как теперь работает рекомендательная система? Она учитывает мою историю просмотров?",\n    "Доступны ли новые функции в приложении на iOS или только на Android?",\n    "Можно ли смотреть прямые трансляции на Smart TV или только через телефон?",\n    "Есть ли планы внедрить функцию совместного просмотра с друзьями?",\n    "Как долго будут храниться записи прошедших трансляций?",\n    "Нужно ли платить за доступ к эксклюзивным шоу или они бесплатные?",\n    "Как я могу отслеживать статус своего предложения по улучшению платформы?"\n  ]\n}'

In [59]:
text = "Почему у вас частная компания? Это является внутренней информацией. Если у вас возникнут вопросы по работе платформы, пожалуйста, пишите нам."

prompt = template + "\n" + text

questions_json = llm.invoke(prompt).text()

In [48]:
text = "Есть ли ограничения на количество получаемых уведомлений? Нет, количество уведомлений не ограничено и зависит от активности на вашем канале."

prompt = template + "\n" + text

llm.invoke(prompt).text()



KeyboardInterrupt: 

In [93]:
text = "Скрипт ответа на нецензурную неконструктивную негативную критику. Мы всегда рады конструктивной критике и очень просим выражаться более корректно. Опишите, пожалуйста, что у вас произошло подробнее. Приложим все усилия для исправления ситуации."

prompt = template + "\n" + text

questions_json = llm.invoke(prompt).text()
questions_json

'```json\n{\n  "questions": [\n    "Какие именно новые функции или обновления были добавлены в Rutube?",\n    "Будут ли новые сериалы или фильмы добавлены в ближайшее время?",\n    "Как теперь можно удобнее смотреть живые трансляции на Rutube?",\n    "Есть ли какие-то ограничения или изменения в доступе к контенту?",\n    "Как я могу сообщить о проблеме или предложить улучшение для платформы?"\n  ]\n}\n```'

In [92]:
questions = json.loads(questions_json)
questions

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [87]:
template_filter = lambda question, answer: f'''
You are a technical support specialist for the Rutube video platform.
The user asked the following question in Russian:
{question}
The chatbot responded with the following answer in Russian:
{answer}

Your task is to evaluate whether the chatbot's response is appropriate and helpful to the user's question.
Follow these rules:
    Answer only "Yes" or "No"

    If the user's question involves internal or confidential information, and the chatbot correctly avoids disclosing it, respond with "Yes" — this is the correct behavior.

    If the question does not involve internal or confidential information, but the chatbot refuses to answer by citing confidentiality, respond with "No" — the response is unnecessarily evasive.

    If the question does not involve internal information, and the chatbot responds accurately, clearly, and helpfully, respond with "Yes".

    Respond with "No" if the chatbot's answer is incorrect, off-topic, unclear, or unhelpful.
'''

In [90]:
answer = "Это является внутренней информацией. Если у вас возникнут вопросы по работе платформы, пожалуйста, пишите нам."
verdicts = []
for question in questions["questions"]:
    print(question)
    prompt = template_filter(question, answer)
    verdict = llm.invoke(prompt).text()
    verdicts.append(verdict)
    print(verdict)

result = zip(verdicts, question)
result

Как статус частной компании влияет на доступный контент и функции платформы?
No
Куда именно нужно писать, если возникнут технические проблемы или вопросы по сервису?
No
Может ли статус частной компании ограничить добавление новых разделов вроде трансляций или подкастов?


KeyboardInterrupt: 